# Portfolio Investment

Budget: $100 M

| level/Fund | 1   | 2   | 3   | 4   | 5   | 6   | 7   | 8   |
|------------|-----|-----|-----|-----|-----|-----|-----|-----|
| 0          | 0   | 0   | 0   | 0   | 0   | 0   | 0   | 0   |
| 1          | 4.1 | 1.8 | 1.5 | 2.2 | 1.3 | 4.2 | 2.2 | 1.0 |
| 2          | 5.8 | 3.0 | 2.5 | 3.8 | 2.4 | 5.9 | 3.5 | 1.7 |
| 3          | 6.5 | 3.9 | 3.3 | 4.8 | 3.2 | 6.6 | 4.2 | 2.3 |
| 4          | 6.8 | 4.5 | 3.8 | 5.5 | 3.9 | 6.8 | 4.6 | 2.8 |

cost to purchase = $10 M * level

remaining funds earns 5% return

In [1]:
import numpy as np
import gurobipy as gp
from gurobipy import GRB

m = gp.Model("portfolio")

REVENUES = np.array([
    [0,   0,   0,   0,   0,   0,   0,   0],
    [4.1, 1.8, 1.5, 2.2, 1.3, 4.2, 2.2, 1.0],
    [5.8, 3.0, 2.5, 3.8, 2.4, 5.9, 3.5, 1.7],
    [6.5, 3.9, 3.3, 4.8, 3.2, 6.6, 4.2, 2.3],
    [6.8, 4.5, 3.8, 5.5, 3.9, 6.8, 4.6, 2.8],
])
COSTS = np.arange(5) * 10

l = m.addMVar(REVENUES.shape, vtype=GRB.BINARY, name="levels")
m.addConstr(l.sum(axis=0) == 1, name="1level")

invested = (l.T @ COSTS).sum()
m.addConstr(invested <= 100, name="budget")

invest_revenue = (l * REVENUES).sum()
savings_revenue = (100 - invested) * 0.05
m.setObjective(invest_revenue + savings_revenue, GRB.MAXIMIZE)

m.optimize()
print(l.X)
print("levels:", l.X.argmax(axis=0))

Set parameter Username
Set parameter LicenseID to value 2780576
Academic license - for non-commercial use only - expires 2027-02-18
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Arch Linux")

CPU model: AMD Ryzen 7 7840HS w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 9 rows, 40 columns and 72 nonzeros (Max)
Model fingerprint: 0xebf75315
Model has 32 linear objective coefficients and an objective constant of -5
Variable types: 0 continuous, 40 integer (40 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+01]
  Objective range  [5e-01, 5e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+02]

Found heuristic solution: objective 5.0000000
Presolve removed 0 rows and 13 columns
Presolve time: 0.00s
Presolved: 9 rows, 27 columns, 54 nonzeros
Variable types: 0 continuous, 27 integer (27 binary)
Found heuristic solution: objective 

In [2]:
# Check.
invested = (l.X.T @ COSTS).sum()
print(invested)

print((l.X * REVENUES).sum() + (100 - invested)*0.05)

100.0
22.299999999999997


Optimal value: $22.3 million.

Optimal solution

fund  | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8
------|---|---|---|---|---|---|---|---
level | 2 | 1 | 1 | 2 | 1 | 2 | 1 | 0

# Cows

81 cows 1-81, produces milk equal to number.

9 sons, same number of cows and same amount of milk.

In [3]:
m = gp.Model("cows")

N_SONS = 9
N_COWS = 81
x = m.addMVar((N_SONS, N_COWS), vtype=GRB.BINARY, name="cows")

m.addConstr(x.sum(axis=0) == 1, name="1owner")

COWS_PER_SON = N_COWS // N_SONS
m.addConstr(x.sum(axis=1) == COWS_PER_SON, name="equalcows")

MILK = np.arange(1, N_COWS+1)
TOTAL_MILK = (N_COWS * (N_COWS+1)) // 2
MILK_PER_SON = TOTAL_MILK // N_SONS
m.addConstr((x @ MILK) == MILK_PER_SON, name="equalcows")

# TODO: think about problem more. can this be made better with an objective?
# TODO: look at cuts used.
m.optimize()
print(x.X)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Arch Linux")

CPU model: AMD Ryzen 7 7840HS w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 99 rows, 729 columns and 2187 nonzeros (Min)
Model fingerprint: 0x2b4dfc8b
Model has 0 linear objective coefficients
Variable types: 0 continuous, 729 integer (729 binary)
Coefficient statistics:
  Matrix range     [1e+00, 8e+01]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+02]

Presolve time: 0.00s
Presolved: 99 rows, 729 columns, 2169 nonzeros
Variable types: 0 continuous, 729 integer (729 binary)

Root relaxation: objective 0.000000e+00, 130 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0    0.00000    0   10

In [4]:
# Check.
print(x.X.sum(axis=1))

milks = np.array([MILK[r > 0.5] for r in x.X])
print(milks)

milk_sums = milks.sum(axis=1)
print(milk_sums)

[9. 9. 9. 9. 9. 9. 9. 9. 9.]
[[ 3  4 28 38 39 48 58 71 80]
 [13 17 24 29 42 43 53 67 81]
 [ 5 18 31 32 35 50 51 69 78]
 [11 15 20 36 44 47 56 63 77]
 [14 21 25 27 40 46 57 64 75]
 [ 2  7 12 34 52 59 61 70 72]
 [ 1 22 30 33 37 41 60 66 79]
 [ 9 10 16 23 45 49 68 73 76]
 [ 6  8 19 26 54 55 62 65 74]]
[369 369 369 369 369 369 369 369 369]


In [5]:
print(f"Each son will have {milk_sums[0]} units of milk.")

print("Cows going to each son:")
for i in range(N_SONS):
    print(f"  son {i}: cows {milks[i]}")

Each son will have 369 units of milk.
Cows going to each son:
  son 0: cows [ 3  4 28 38 39 48 58 71 80]
  son 1: cows [13 17 24 29 42 43 53 67 81]
  son 2: cows [ 5 18 31 32 35 50 51 69 78]
  son 3: cows [11 15 20 36 44 47 56 63 77]
  son 4: cows [14 21 25 27 40 46 57 64 75]
  son 5: cows [ 2  7 12 34 52 59 61 70 72]
  son 6: cows [ 1 22 30 33 37 41 60 66 79]
  son 7: cows [ 9 10 16 23 45 49 68 73 76]
  son 8: cows [ 6  8 19 26 54 55 62 65 74]
